In [1]:
pip install datasets

In [2]:
from datasets import load_dataset

dataset_name = 'Chiranjeev007/STL-10_Subset'
dataset = load_dataset(dataset_name)

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/723 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/88.9M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/8.82M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 500
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
})


In [3]:
train_ds = dataset["train"]
valid_ds = dataset["validation"]
test_ds  = dataset["test"]

print(train_ds)
print(valid_ds)
print(test_ds)

Dataset({
    features: ['image', 'label'],
    num_rows: 5000
})
Dataset({
    features: ['image', 'label'],
    num_rows: 500
})
Dataset({
    features: ['image', 'label'],
    num_rows: 1000
})


###Importing torchvision for pretrained models


In [4]:
# Install required libraries
!pip install torchvision -q

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [5]:
# Define training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((96, 96)),                 # Resize images
    transforms.RandomHorizontalFlip(),           # Random flip
    transforms.RandomRotation(20),               # Random rotation
    transforms.ColorJitter(0.3,0.3,0.3,0.1),     # Color augmentation
    transforms.ToTensor(),                       # Convert to tensor
    transforms.Normalize([0.5]*3, [0.5]*3)       # Normalize
])

In [6]:
# Define validation/test transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [7]:
# Custom Dataset wrapper for HuggingFace dataset
class CustomSTL10(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image = self.dataset[idx]["image"]  # PIL image
        label = self.dataset[idx]["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

In [8]:
# Create PyTorch dataset objects
train_dataset = CustomSTL10(train_ds, transform=train_transform)
val_dataset   = CustomSTL10(valid_ds, transform=test_transform)
test_dataset  = CustomSTL10(test_ds,  transform=test_transform)

In [9]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

In [10]:
# Verify DataLoader
images, labels = next(iter(train_loader))
print(images.shape)  # Expected: [32, 3, 96, 96]
print(labels.shape)  # Expected: [32]

torch.Size([32, 3, 96, 96])
torch.Size([32])


In [11]:
# Install required libraries
!pip install wandb torchvision -q

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import wandb

In [13]:
# Initialize Weights & Biases
wandb.init(project="stl10-resnet18", name="resnet18-finetune")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: m25csa019 (m25csa019-iit-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [14]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
# Load pretrained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Modify final layer (STL-10 has 10 classes)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 200MB/s]


In [16]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [17]:
# Accuracy function
def calculate_accuracy(outputs, labels):
    _, preds = torch.max(outputs, 1)
    correct = (preds == labels).sum().item()
    return correct / labels.size(0)

In [18]:
# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    train_acc = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_acc += calculate_accuracy(outputs, labels)

    train_loss /= len(train_loader)
    train_acc /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    val_acc = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            val_acc += calculate_accuracy(outputs, labels)

    val_loss /= len(val_loader)
    val_acc /= len(val_loader)

    # Log to WandB
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

Epoch [1/10] Train Loss: 1.2315, Train Acc: 0.5772, Val Loss: 0.8819, Val Acc: 0.6691
Epoch [2/10] Train Loss: 0.8653, Train Acc: 0.6996, Val Loss: 0.9114, Val Acc: 0.6906
Epoch [3/10] Train Loss: 0.7885, Train Acc: 0.7381, Val Loss: 0.8592, Val Acc: 0.7039
Epoch [4/10] Train Loss: 0.7129, Train Acc: 0.7566, Val Loss: 1.0380, Val Acc: 0.6797
Epoch [5/10] Train Loss: 0.6059, Train Acc: 0.7972, Val Loss: 0.8723, Val Acc: 0.7059
Epoch [6/10] Train Loss: 0.6115, Train Acc: 0.7932, Val Loss: 0.7712, Val Acc: 0.7516
Epoch [7/10] Train Loss: 0.5243, Train Acc: 0.8240, Val Loss: 1.1901, Val Acc: 0.6707
Epoch [8/10] Train Loss: 0.5085, Train Acc: 0.8350, Val Loss: 0.7372, Val Acc: 0.7551
Epoch [9/10] Train Loss: 0.4606, Train Acc: 0.8465, Val Loss: 0.6579, Val Acc: 0.8012
Epoch [10/10] Train Loss: 0.4246, Train Acc: 0.8587, Val Loss: 0.7035, Val Acc: 0.7980


In [19]:
# Install required library
!pip install huggingface_hub -q

In [20]:
import torch
from huggingface_hub import login, HfApi

In [21]:
# Login to Hugging Face (paste your HF token)
login()

In [22]:
best_val_acc = 0.0
best_model_path = "best_resnet18_stl10.pth"

for epoch in range(num_epochs):

    # Training
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print("Best model updated and saved.")

Epoch [1/10] Train Loss: 0.3995, Train Acc: 0.8662, Val Loss: 0.7330, Val Acc: 0.7700
Best model updated and saved.
Epoch [2/10] Train Loss: 0.4050, Train Acc: 0.8622, Val Loss: 0.5818, Val Acc: 0.8200
Best model updated and saved.
Epoch [3/10] Train Loss: 0.3155, Train Acc: 0.8922, Val Loss: 0.7402, Val Acc: 0.7680
Epoch [4/10] Train Loss: 0.3344, Train Acc: 0.8860, Val Loss: 0.7348, Val Acc: 0.7500
Epoch [5/10] Train Loss: 0.3271, Train Acc: 0.8934, Val Loss: 0.7056, Val Acc: 0.8200
Epoch [6/10] Train Loss: 0.2941, Train Acc: 0.9034, Val Loss: 0.7258, Val Acc: 0.7820
Epoch [7/10] Train Loss: 0.2541, Train Acc: 0.9152, Val Loss: 0.7055, Val Acc: 0.8200
Epoch [8/10] Train Loss: 0.2538, Train Acc: 0.9122, Val Loss: 0.6649, Val Acc: 0.7920
Epoch [9/10] Train Loss: 0.2693, Train Acc: 0.9068, Val Loss: 0.5878, Val Acc: 0.8460
Best model updated and saved.
Epoch [10/10] Train Loss: 0.3155, Train Acc: 0.8940, Val Loss: 0.7538, Val Acc: 0.7600


In [24]:
repo_id = "nthapa/resnet18-stl10-best"

api = HfApi()
api.create_repo(repo_id=repo_id, exist_ok=True)

RepoUrl('https://huggingface.co/nthapa/resnet18-stl10-best', endpoint='https://huggingface.co', repo_type='model', repo_id='nthapa/resnet18-stl10-best')

In [25]:
api.upload_file(
    path_or_fileobj="best_resnet18_stl10.pth",
    path_in_repo="pytorch_model.bin",
    repo_id=repo_id,
    repo_type="model"
)

print("Best model pushed successfully 🚀")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  best_resnet18_stl10.pth     :   1%|1         |  562kB / 44.8MB            

Best model pushed successfully 🚀
